# Cosmo DL — TCN на замороженных фолдах ML (C-03)

Notebook **не создаёт** folds, MaskSpec или submission. Он исполняет DL runner
поверх производного C-03 manifest: outer keys, censoring и baseline OOF пришли
из опубликованной ветки ML, DL добавляет только train-блоки и inner-монитор.

Порядок: контроль датасета → offline tests → GPU smoke → полная CV → архив.
Каждая ячейка падает громко: молчаливого отката на CPU или на свои folds нет.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Единственное, что нужно править: имя добавленного Kaggle Dataset.
BUNDLE = Path("/kaggle/input/cosmo-dl-bundle")
OUTPUT = Path("/kaggle/working/dl_tcn_run")

manifest = json.loads((BUNDLE / "BUNDLE_MANIFEST.json").read_text(encoding="utf-8"))
FOLD_MANIFEST = BUNDLE / "inputs/dl_c03.json"
assert FOLD_MANIFEST.is_file(), "нет производного C-03 manifest"

# До слияния веток код лежит в src_dl/ и src_ml/, после слияния — в одном src/.
layout = manifest.get("layout", "split")
roots = ["src"] if layout == "merged" else ["src_dl", "src_ml"]
for name in roots:
    assert (BUNDLE / name / "veg_recovery").is_dir(), f"нет {name}/veg_recovery"
assert (BUNDLE / roots[0] / "veg_recovery/dl/train.py").is_file(), "нет кода DL"

env = dict(
    os.environ,
    PYTHONPATH=os.pathsep.join(str(BUNDLE / name) for name in roots),
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)

def run(*args, cwd=BUNDLE):
    subprocess.run([sys.executable, *args], cwd=str(cwd), env=env, check=True)

print("раскладка:", layout, "| PYTHONPATH:", env["PYTHONPATH"])
print("DL commit:", manifest["dl_commit"], "| ML commit:", manifest["ml_commit"])
print("C-03 review status:", manifest["c03_review_status"])
print("файлов в bundle:", len(manifest["files"]))
print("команда полного прогона:", manifest["run_command"])


In [ ]:
# Preflight: хеши входов, равенство метрик ML baseline, отсутствие пересечений
# train/inner/outer и наличие producer censoring. Torch здесь не импортируется.
run("-m", "veg_recovery.dl.train", "--fold-manifest", str(FOLD_MANIFEST),
    "--preflight-only")


In [ ]:
import torch
assert torch.cuda.is_available(), "включите GPU в настройках Kaggle Notebook"
print("GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__,
      "| CUDA:", torch.version.cuda)
run("-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/dl", "tests/anomalies")
run("-m", "veg_recovery.dl.train", "--smoke", "--device", "cuda", "--epochs", "3",
    "--window", "15", "--hidden-size", "16", "--layers", "2",
    "--output", "/kaggle/working/dl_gpu_smoke")


In [ ]:
# Полная CV: 16 фолдов ML x 3 seed. epoch-policy=fixed не выбирает epoch
# по outer OOF; inner RMSE логируется как монитор. Бюджет 16 закреплён заранее
# по inner-кривой пилота: плато на 12-17 эпохе во всех проверенных режимах.
run("-m", "veg_recovery.dl.train", "--fold-manifest", str(FOLD_MANIFEST),
    "--device", "cuda", "--seeds", "17", "42", "73",
    "--window", "61", "--epochs", "16", "--epoch-policy", "fixed",
    "--base-mode", "anchored", "--batch-size", "64",
    "--output", str(OUTPUT))


In [ ]:
import shutil
report = json.loads((OUTPUT / "cv_report.json").read_text(encoding="utf-8"))
for seed in report["seeds"]:
    print(seed["seed"], "DL composite", seed["dl"]["composite_rmse"],
          "| ML composite", seed["ml"]["composite_rmse"],
          "| base-only", seed["dl_base_only"]["composite_rmse"])
    print("   по режимам DL:", seed["dl"]["split_rmse"])
    print("   bootstrap по полигонам:", seed["polygon_bootstrap"])
archive = shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT)
print("скачайте:", archive)


In [ ]:
# Ablation: та же CV с линейной base вместо сезонного prior. Отделяет вклад
# сети от вклада base. Запускать только после основного прогона.
run("-m", "veg_recovery.dl.train", "--fold-manifest", str(FOLD_MANIFEST),
    "--device", "cuda", "--seeds", "17", "--window", "61", "--epochs", "16",
    "--epoch-policy", "fixed", "--base-mode", "linear", "--batch-size", "64",
    "--output", "/kaggle/working/dl_tcn_linear_base")
